# Laboratory Exercise — Image Classification Using Pre-trained Models

**Course:** DS Elective 4 | **Portfolio:** Bear's Data Den  
**Student:** Roque Kyle Mallari  
**Task:** Evaluate five pre-trained TorchVision classifiers on a curated image-classification dataset.

> **Run this notebook from top to bottom and save the notebook after execution.** The outputs, charts, metrics, and automatically generated interpretation will then appear on your Jupyter Book website. This notebook intentionally contains no made-up performance scores; the models must be evaluated on your computer first.

### Learning outcomes
1. Select and organize labeled image data.
2. Write a custom `torch.utils.data.Dataset` and build `DataLoader` objects.
3. Apply the correct pre-processing associated with each model's pre-trained weights.
4. Run inference with **five** TorchVision models without fine-tuning.
5. Compare **accuracy, precision, recall, and F1-score** and interpret the results.

## 1. Dataset selection and task definition

The **Hymenoptera Ants vs. Bees** dataset is distributed with the [official PyTorch transfer-learning tutorial](https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial.html). It includes two folders (`ants`, `bees`) and separate `train` and `val` splits. This lab uses the **validation split only** to evaluate already-trained ImageNet classifiers: **no model is trained or fine-tuned**.

**Why this dataset?** ImageNet's 1,000 categories include `ant` and `bee`. That means we can compare pre-trained models on a meaningful, tightly-defined **two-choice inference task**.

**Important evaluation rule:** ImageNet models output 1,000 scores. For our *closed-set* ant-vs-bee classification, we compare the two scores for the ImageNet categories named `ant` and `bee`, and pick the larger one. This does **not** mean that the model's unrestricted ImageNet top-1 answer was `ant` or `bee`; other ImageNet categories are excluded from this decision. All models use the same rule, so the comparison is consistent. The dataset was not gathered by the student.

The notebook automatically downloads the data from the official source. If automatic download fails, manually extract [hymenoptera_data.zip](https://download.pytorch.org/tutorial/hymenoptera_data.zip) into `data/` alongside this notebook (folder layout is shown below).

### Dataset folder organization

```text
labs/
  Lab_Image_Classification_Pretrained_Models.ipynb
  data/
    hymenoptera_data/
      train/
        ants/ ...
        bees/ ...
      val/
        ants/ ...
        bees/ ...
```

`val/` supplies the labeled evaluation images. `train/` is retained from the official download but not used to fit or choose any model. We check file validity and report class balance before evaluation.

**Installation (once, in Command Prompt):** `python -m pip install torch torchvision pillow numpy pandas matplotlib scikit-learn ipython`

> On Windows, open the Jupyter Notebook or JupyterLab in the `labs` folder before running this notebook. The five models will download ImageNet weights during their first use (requires an internet connection).

In [ ]:
from pathlib import Path
from collections import Counter
from urllib.request import urlretrieve
import time
import zipfile
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, UnidentifiedImageError

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
from IPython.display import display, Markdown

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 16
NUM_WORKERS = 0   # Windows-friendly; change only if your system supports multiprocessing
MAX_PER_CLASS = None  # None evaluates ALL available validation images; 30 = fast test subset

print('PyTorch:', torch.__version__)
print('Device:', DEVICE)
print('Evaluation subset per class:', MAX_PER_CLASS or 'All validation images')

## 2. Download, curate, and inspect the images

The data are stored in **class-specific folders**. Each image is checked for readability and convertible RGB format; corrupted or unrelated files are skipped. This ensures the custom dataset is suitable for PyTorch evaluation.

In [ ]:
DATA_DIR = Path('data')
ROOT = DATA_DIR / 'hymenoptera_data'
ZIP_PATH = DATA_DIR / 'hymenoptera_data.zip'
DATA_URL = 'https://download.pytorch.org/tutorial/hymenoptera_data.zip'
DATA_DIR.mkdir(parents=True, exist_ok=True)

if not (ROOT / 'val' / 'ants').is_dir() or not (ROOT / 'val' / 'bees').is_dir():
    if not ZIP_PATH.exists():
        print('Downloading the official PyTorch dataset (first run only)...')
        try:
            urlretrieve(DATA_URL, ZIP_PATH)
        except Exception as exc:
            ZIP_PATH.unlink(missing_ok=True)
            raise RuntimeError(
                'Dataset download failed. Manually download '+ DATA_URL +
                ' and extract it inside the labs/data folder, then rerun this cell.'
            ) from exc
    print('Extracting dataset...')
    # Validate paths to prevent archive entries escaping the data directory.
    with zipfile.ZipFile(ZIP_PATH) as archive:
        target = DATA_DIR.resolve()
        for entry in archive.infolist():
            extracted = (DATA_DIR / entry.filename).resolve()
            if extracted != target and target not in extracted.parents:
                raise ValueError('Unsafe path in dataset archive: ' + entry.filename)
        archive.extractall(DATA_DIR)

assert (ROOT / 'val' / 'ants').is_dir() and (ROOT / 'val' / 'bees').is_dir(), (
    'Missing class folders. Expected data/hymenoptera_data/val/{ants,bees}.'
)
print('Dataset folder:', ROOT.resolve())

In [ ]:
CLASSES = ['ants', 'bees']
CLASS_TO_INDEX = {'ants': 0, 'bees': 1}
ALLOWED_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}


def curate_samples(split='val', max_per_class=None):
    samples, rejected = [], []
    for label_name in CLASSES:
        folder = ROOT / split / label_name
        if not folder.is_dir():
            raise FileNotFoundError(f'Missing class folder: {folder}')
        valid_class = []
        for path in sorted(folder.iterdir()):
            if path.suffix.lower() not in ALLOWED_EXTENSIONS or not path.is_file():
                continue
            try:
                with Image.open(path) as img:
                    img.verify()
                with Image.open(path) as img:
                    img.convert('RGB')
            except (OSError, ValueError, UnidentifiedImageError):
                rejected.append(path.name)
                continue
            valid_class.append((path, CLASS_TO_INDEX[label_name]))
        if max_per_class is not None:
            rng = random.Random(SEED)
            valid_class = sorted(rng.sample(valid_class, min(max_per_class, len(valid_class))))
        samples.extend(valid_class)
    return samples, rejected

EVAL_SAMPLES, REJECTED = curate_samples('val', MAX_PER_CLASS)
print('Curated validation images:', len(EVAL_SAMPLES))
print('Unreadable images skipped:', len(REJECTED))
print('Class counts:', Counter(CLASSES[label] for _, label in EVAL_SAMPLES))
assert len(EVAL_SAMPLES) > 0 and len({label for _,label in EVAL_SAMPLES}) == 2, 'Both classes must be present.'

distribution = pd.DataFrame([
    {'Class': label_name, 'Validation images': sum(lab == idx for _,lab in EVAL_SAMPLES)}
    for idx, label_name in enumerate(CLASSES)
])
display(distribution)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(distribution['Class'], distribution['Validation images'])
ax.set(ylabel='Number of images', title='Curated validation dataset: class balance')
for i, n in enumerate(distribution['Validation images']):
    ax.text(i, n + 1, str(n), ha='center')
plt.tight_layout()
plt.show()

In [ ]:
# Show representative validation images (actual dataset photos, not generated examples).
fig, axs = plt.subplots(2, 4, figsize=(11, 6))
for class_id in (0, 1):
    items = [path for path, lab in EVAL_SAMPLES if lab == class_id][:4]
    for col, ax in enumerate(axs[class_id]):
        ax.axis('off')
        if col < len(items):
            with Image.open(items[col]) as im:
                ax.imshow(im.convert('RGB'))
            ax.set_title(CLASSES[class_id])
fig.suptitle('Sample images from the curated Ants vs. Bees dataset', y=1.02)
plt.tight_layout()
plt.show()

## 3. Custom PyTorch Dataset and DataLoader

The custom `Dataset` returns an `(image_tensor, numeric_label)` pair. Different models may require different resize, crop, or normalization settings, so the **pretrained weight's own `.transforms()`** is passed to the dataset each time. All models receive the **same original validation images**, ensuring a fair comparison.

The DataLoader batches validation images without shuffling so predictions always match the ground-truth labels.

In [ ]:
class AntBeeImageDataset(Dataset):
    """Custom image classification Dataset for labeled ant/bee photo paths."""
    def __init__(self, samples, transform):
        self.samples = list(samples)
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        image_path, label = self.samples[index]
        with Image.open(image_path) as image:
            image = image.convert('RGB')
            image = self.transform(image)
        return image, label


def make_eval_loader(weight_enum):
    dataset = AntBeeImageDataset(EVAL_SAMPLES, transform=weight_enum.transforms())
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == 'cuda'))

# Verify that Dataset and DataLoader produce properly shaped tensors.
probe_weights = models.ResNet18_Weights.DEFAULT
probe_loader = make_eval_loader(probe_weights)
images, targets = next(iter(probe_loader))
print('Batch images:', tuple(images.shape), '(N, C, H, W)')
print('Batch labels:', tuple(targets.shape))
print('Label lookup:', CLASS_TO_INDEX)
assert images.shape[1] == 3, 'Models require 3-channel images.'

## 4. Select five pre-trained models

We compare five ImageNet-pre-trained models from `torchvision.models`:

| Model | Architecture family | Useful distinction |
|---|---|---|
| ResNet-18 | Residual CNN | Skip connections help deep feature learning |
| MobileNetV3-Small | Mobile / efficient CNN | Smaller, lightweight architecture |
| EfficientNet-B0 | Compound-scaled CNN | Balances depth, width, and image resolution |
| SqueezeNet 1.1 | Compact CNN | Small parameter count with Fire modules |
| ShuffleNetV2 1.0× | Lightweight CNN | Channel shuffle and efficiency-focused blocks |

Models are used with their **published ImageNet weights**; there is **no fine-tuning on ants or bees**. All five weight definitions have `ant` and `bee` among their ImageNet categories. The first run downloads each model's parameters. Model loading/download time is not included in inference time.

In [ ]:
MODEL_SPECS = [
    ('ResNet-18', models.resnet18, models.ResNet18_Weights.DEFAULT),
    ('MobileNetV3-Small', models.mobilenet_v3_small, models.MobileNet_V3_Small_Weights.DEFAULT),
    ('EfficientNet-B0', models.efficientnet_b0, models.EfficientNet_B0_Weights.DEFAULT),
    ('SqueezeNet1.1', models.squeezenet1_1, models.SqueezeNet1_1_Weights.DEFAULT),
    ('ShuffleNetV2 x1.0', models.shufflenet_v2_x1_0, models.ShuffleNet_V2_X1_0_Weights.DEFAULT),
]

for name, factory, weight in MODEL_SPECS:
    cats = weight.meta['categories']
    assert 'ant' in cats and 'bee' in cats, f'Missing ant / bee ImageNet label in {name}'
    print(f'{name}: ant ImageNet index={cats.index("ant")}, bee ImageNet index={cats.index("bee")}')

## 5. Perform inference and calculate evaluation metrics

**Inference rule:** Each model produces 1,000 output logits. We extract the `ant` and `bee` logits and choose the larger one. The class scores do not need softmax for argmax selection, since softmax preserves rank.

Metrics (macro averaging treats ants and bees equally):

- **Accuracy:** fraction of correctly classified images.
- **Precision:** how often predicted class membership was correct.
- **Recall:** how many real examples of a class were found.
- **F1-score:** harmonic mean of precision and recall.

We measure inference time only after model weights have loaded. The timing depends on the current hardware, so it is not a universal speed benchmark.

In [ ]:
results = []
prediction_store = {}
ground_truth = np.array([label for _, label in EVAL_SAMPLES], dtype=int)

for model_name, create_model, pretrained_weights in MODEL_SPECS:
    print(f'\nEvaluating {model_name} ...')
    model = create_model(weights=pretrained_weights).to(DEVICE)
    model.eval()
    loader = make_eval_loader(pretrained_weights)

    # Two ImageNet category indexes, aligned to binary labels: 0=ants, 1=bees.
    category_names = pretrained_weights.meta['categories']
    image_net_indices = [category_names.index('ant'), category_names.index('bee')]
    predicted = []

    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    tic = time.perf_counter()
    with torch.inference_mode():
        for batch_images, _ in loader:
            batch_images = batch_images.to(DEVICE)
            logits = model(batch_images)        # [batch, 1000] ImageNet class scores
            binary_logits = logits[:, image_net_indices]
            predicted.extend(binary_logits.argmax(dim=1).cpu().numpy().tolist())
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - tic

    predicted = np.array(predicted, dtype=int)
    assert len(predicted) == len(ground_truth)
    acc = accuracy_score(ground_truth, predicted)
    precision, recall, f1, _ = precision_recall_fscore_support(
        ground_truth, predicted, labels=[0, 1], average='macro', zero_division=0
    )
    count_params_m = sum(p.numel() for p in model.parameters()) / 1_000_000
    results.append({
        'Model': model_name, 'Accuracy': acc, 'Precision': precision,
        'Recall': recall, 'F1-score': f1, 'Parameters (M)': count_params_m,
        'Inference seconds': elapsed
    })
    prediction_store[model_name] = predicted
    print(f'  Accuracy={acc:.3f}; Macro F1={f1:.3f}; Parameters={count_params_m:.2f}M')
    del model
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

comparison_df = pd.DataFrame(results).sort_values(
    ['F1-score', 'Accuracy'], ascending=False
).reset_index(drop=True)

pd.set_option('display.max_columns', None)
display(comparison_df.round(4))

# Export measured data for your portfolio/report.
OUTPUT_DIR = Path('results')
OUTPUT_DIR.mkdir(exist_ok=True)
comparison_df.to_csv(OUTPUT_DIR / 'pretrained_model_comparison.csv', index=False)
print('Saved:', OUTPUT_DIR / 'pretrained_model_comparison.csv')

## 6. Model comparison visualizations

In [ ]:
metric_columns = ['Accuracy', 'Precision', 'Recall', 'F1-score']
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(comparison_df))
bar_width = 0.18
for idx, metric in enumerate(metric_columns):
    bars = ax.bar(x + (idx - 1.5) * bar_width, comparison_df[metric],
                  width=bar_width, label=metric)
ax.set_xticks(x)
ax.set_xticklabels(comparison_df['Model'], rotation=15, ha='right')
ax.set_ylim(0, 1.08)
ax.set_ylabel('Score (0 to 1)')
ax.set_title('Five ImageNet pre-trained models: Ants vs. Bees (validation images)')
ax.legend(ncol=4, loc='upper center', bbox_to_anchor=(0.5, -0.26))
ax.grid(axis='y', alpha=0.2)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'model_comparison.png', dpi=160, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(comparison_df['Model'][::-1], comparison_df['Parameters (M)'][::-1])
ax.set(xlabel='Number of parameters (millions)', title='Model complexity comparison')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'parameter_comparison.png', dpi=160, bbox_inches='tight')
plt.show()

## 7. Best-performing model: confusion matrix and per-class scores

The model with the highest **macro F1-score** is highlighted, with accuracy as the tiebreaker. Selecting the highest-scoring model **on the validation split** is exploratory; for a final unbiased performance estimate you would need a separate unseen test set. This exercise reports validation results and does not claim independent test accuracy.

In [ ]:
best_model = comparison_df.loc[0, 'Model']
best_preds = prediction_store[best_model]
cm = confusion_matrix(ground_truth, best_preds, labels=[0,1])

fig, ax = plt.subplots(figsize=(5.8, 4.9))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0,1], labels=CLASSES)
ax.set_yticks([0,1], labels=CLASSES)
ax.set(xlabel='Predicted class', ylabel='Actual class', title=f'{best_model}: Confusion Matrix')
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=15,
                color='white' if cm[i,j] > (cm.max() / 2) else 'black')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'best_model_confusion_matrix.png', dpi=160, bbox_inches='tight')
plt.show()

print('Best model by macro F1:', best_model)
print(classification_report(ground_truth, best_preds, labels=[0,1],
                            target_names=CLASSES, zero_division=0))

## 8. Review example predictions

This gallery prioritizes the model's incorrect classifications so you can inspect what made some images challenging (e.g., tiny insects, cluttered backgrounds, partial views).

In [ ]:
mistake_indices = [i for i, (y, pred) in enumerate(zip(ground_truth, best_preds)) if y != pred]
correct_indices = [i for i, (y, pred) in enumerate(zip(ground_truth, best_preds)) if y == pred]
chosen = (mistake_indices[:6] + correct_indices[:max(0, 6 - len(mistake_indices[:6]))])[:6]

if not chosen:
    print('No images to display.')
else:
    fig, axes = plt.subplots(2, 3, figsize=(10.5, 7))
    for ax, idx in zip(axes.flat, chosen):
        path, truth = EVAL_SAMPLES[idx]
        with Image.open(path) as pic:
            ax.imshow(pic.convert('RGB'))
        pred = int(best_preds[idx])
        ax.set_title(f'True: {CLASSES[truth]} | Predicted: {CLASSES[pred]}', fontsize=10)
        ax.axis('off')
    for ax in list(axes.flat)[len(chosen):]:
        ax.axis('off')
    plt.suptitle(f'Sample predictions for {best_model}', y=1.02)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'sample_predictions.png', dpi=160, bbox_inches='tight')
    plt.show()

## 9. Short analysis and conclusion

**The following cell generates an interpretation using your actual measured scores.** It does not insert fictional values, and the interpretation changes if the results change. After running it, review the explanation and add a personal observation about the sample images or errors that you noticed.

In [ ]:
winner = comparison_df.iloc[0]
last = comparison_df.iloc[-1]
score_gap = winner['F1-score'] - last['F1-score']
smallest = comparison_df.sort_values('Parameters (M)').iloc[0]
largest = comparison_df.sort_values('Parameters (M)', ascending=False).iloc[0]

analysis = f"""### Analysis

We evaluated **{len(comparison_df)} ImageNet-pre-trained models** on **{len(ground_truth)} curated validation images** ({sum(ground_truth==0)} ants and {sum(ground_truth==1)} bees) using the same two-choice prediction rule. **{winner['Model']}** achieved the highest macro F1-score (**{winner['F1-score']:.3f}**) and an accuracy of **{winner['Accuracy']:.3f}**. The lowest F1-score was **{last['F1-score']:.3f}** from **{last['Model']}**, giving a difference of **{score_gap:.3f}**.

Different performance levels can result from **architecture**, **model capacity**, learned **ImageNet feature representations**, and each model's recommended **input preprocessing**. The smallest model in this comparison has approximately **{smallest['Parameters (M)']:.2f} million parameters** ({smallest['Model']}), while the largest has **{largest['Parameters (M)']:.2f} million** ({largest['Model']}). A larger network is not automatically more accurate on a small two-class dataset. Insect size, background clutter, photography angles, and the small validation sample may also affect the results. These were **pretrained zero-fine-tuning comparisons**, not separately trained ant-vs-bee classifiers.

### Conclusion

For this **ants-vs-bees closed-set validation experiment**, **{winner['Model']}** was the strongest of the five evaluated models by macro F1-score. However, the best choice for deployment would also consider memory, model size, and inference speed. Results are limited to this dataset and two-choice label mapping; they should not be interpreted as full ImageNet classification accuracy or performance on other tasks. A useful next step is to test a separate held-out dataset or fine-tune the classifiers on new training data, then compare results fairly.
"""
display(Markdown(analysis))
(OUTPUT_DIR / 'analysis_and_conclusion.md').write_text(analysis, encoding='utf-8')
print('Saved:', OUTPUT_DIR / 'analysis_and_conclusion.md')

## 10. Reflection, reproducibility, and limitations

- **Custom implementation:** Wrote `AntBeeImageDataset` rather than relying on `ImageFolder`; tested it through `DataLoader` batches.
- **Pretrained model comparison:** Used ResNet-18, MobileNetV3-Small, EfficientNet-B0, SqueezeNet1.1, and ShuffleNetV2 x1.0 with the official TorchVision pretrained weights and model-specific transforms.
- **Metrics:** Accuracy, macro precision, macro recall, and macro F1-score were computed against the curated validation labels. Also reported parameter counts and runtime-specific inference seconds.
- **Correct interpretation:** This is **two-choice ImageNet logit comparison**, not training, not fine-tuning, and not unconstrained 1,000-class top-1 accuracy. Do not claim results until you have actually run all five models.
- **Data quality:** Check the class counts, sample images, and rejected files printed above. Dataset images come from the official PyTorch tutorial; no personal dataset was collected.
- **Portfolio publishing:** The notebook needs to be executed and **saved with its outputs** before building Jupyter Book, because the project's build configuration intentionally does not execute notebooks.

**Student reflection (write after running):** Which misclassified images were most difficult? Did a small model perform almost as well as a large one? What would you improve next time?

The most difficult images to classify were likely those where the ants or bees appeared small, blurry, partially hidden, or surrounded by complex backgrounds. These conditions make it harder for the models to recognize important features.

The smaller models may perform almost as well as the larger models, depending on their accuracy and F1-scores. This shows that a more complex model does not always guarantee better performance, especially for a simple classification task.

Next time, I would improve the dataset by adding more diverse images, applying data augmentation, and fine-tuning the pretrained models to achieve better classification results.